# 03 · Baseline Forecasts – Seasonal Naïve & SARIMA
We start modelling with two simple baselines on a **hand-picked subset of wards** (the five most-burglary wards city-wide):
* **Seasonal-naïve**: predict next month will equal the same month last year.
* **SARIMA(1,0,1)(1,0,1)[12]**: a quick seasonal ARIMA fitted per ward.

Evaluation metric → **MAE** on the *hold-out* year 2024 (last 12 points).

In [1]:
%pip install pandas numpy matplotlib seaborn statsmodels Jinja2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pathlib, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from statsmodels.tsa.statespace.sarimax import SARIMAX

ROOT = pathlib.Path('..').resolve()  # assumes notebook lives in notebooks/
PANEL = ROOT / 'data_cache/processed/ward_month_burglary.parquet'
df = pd.read_parquet(PANEL)
df.head()

,Month,WD24CD,WD24NM,burglaries
0,2010-12-01,E05009317,Bethnal Green East,5
1,2010-12-01,E05009318,Blackwall & Cubitt Town,1
2,2010-12-01,E05009319,Bow East,7
3,2010-12-01,E05009320,Bow West,7
4,2010-12-01,E05009321,Bromley North,4


## Pick top-5 burglary wards

In [3]:
top5_codes = (df.groupby('WD24CD')['burglaries'].sum()
               .nlargest(5).index.tolist())
top5_codes

['E05013808', 'E05013806', 'E05013801', 'E05013653', 'E05013662']

In [4]:
top5 = df[df['WD24CD'].isin(top5_codes)].copy()
# pivot to wide form (index=Month) for easy slicing per ward
wide = top5.pivot(index='Month', columns='WD24CD', values='burglaries').sort_index()
wide.tail()

WD24CD,E05013653,E05013662,E05013801,E05013806,E05013808
Month,,,,,
2024-10-01,34,10,24,25,60
2024-11-01,30,18,31,29,59
2024-12-01,20,19,38,36,65
2025-01-01,27,16,21,18,69
2025-02-01,21,16,34,18,32


## Train / test split
* **Train**: Dec 2013 – Dec 2023
* **Test** : Jan 2024 – Dec 2024 (12 points)

In [5]:
train = wide.loc[:'2023-12']
test  = wide.loc['2024-01':]

## 1 · Seasonal-naïve baseline (12-month lag)

In [6]:
pred_sn = wide.shift(12).loc[test.index]  # simply last year's same month
mae_sn = (pred_sn - test).abs().mean()
mae_sn.to_frame('MAE').style.format('{:.2f}')

,MAE
WD24CD,
E05013653,8.57
E05013662,4.00
E05013801,8.86
E05013806,7.86
E05013808,17.57


## 2 · SARIMA(1,0,1)(1,0,1)[12] per ward

In [7]:
mae_sarima = {}
for code in top5_codes:
    y_train = train[code]
    model = SARIMAX(y_train, order=(1,0,1), seasonal_order=(1,0,1,12), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    forecast = model.get_forecast(steps=len(test)).predicted_mean
    mae = (forecast.values - test[code].values).mean().__abs__()
    mae_sarima[code] = mae
mae_sarima = pd.Series(mae_sarima, name='MAE')
mae_sarima.to_frame().style.format('{:.2f}')

c:\Users\Ergi Livanaj\Desktop\University\Year 2\Quarter 4\4CBLW00-20 (JBG050 Data Challenge 2)\4CBLW00-20-Group-34\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Ergi Livanaj\Desktop\University\Year 2\Quarter 4\4CBLW00-20 (JBG050 Data Challenge 2)\4CBLW00-20-Group-34\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Ergi Livanaj\Desktop\University\Year 2\Quarter 4\4CBLW00-20 (JBG050 Data Challenge 2)\4CBLW00-20-Group-34\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  r

,MAE
E05013808,15.27
E05013806,2.90
E05013801,5.70
E05013653,2.36
E05013662,1.88


### Compare baselines

In [8]:
pd.DataFrame({'Seasonal-naïve': mae_sn, 'SARIMA': mae_sarima})

,Seasonal-naïve,SARIMA
E05013653,8.571429,2.359318
E05013662,4.000000,1.881631
E05013801,8.857143,5.696500
E05013806,7.857143,2.900709
E05013808,17.571429,15.265671


*Interpretation*: SARIMA generally beats the seasonal-naïve MAE (lower is better) on our five busiest wards. This justifies using SARIMA or more sophisticated models going forward.